# Task 1 Statistics Evidence - xfan0282


## Setup

This notebook shows the five Task 1 derived statistics for `xfan0282`. Before running it, make sure `uv sync` has been run and the project virtual environment is selected as the notebook kernel.

The setup cell changes the working directory to the project root. This makes the raw CSV path work even when this notebook is opened from the `notebooks/xfan0282` folder.


In [ ]:
from importlib import import_module
import os

import pandas as pd

from data2001.common.paths import PROJECT_ROOT, resolve_project_path
from data2001.config import load_settings
from data2001.task1_cleaning.workflow import run_task1_cleaning


os.chdir(PROJECT_ROOT)

MEMBER_UNIKEY = "xfan0282"
settings = load_settings("configs/local.yaml")
statistics_module = import_module(f"data2001.task1_statistics.{MEMBER_UNIKEY}_statistics")

member_context = pd.DataFrame([
    {
        "unikey": MEMBER_UNIKEY,
        "project_root": str(PROJECT_ROOT),
        "raw_task1_csv": str(resolve_project_path(settings.outputs.raw_task1_csv)),
        "processed_task1_cleaned_csv": str(resolve_project_path(settings.outputs.processed_task1_cleaned_csv)),
    }
])
display(member_context)

## Shared Cleaning Input

This section runs the Task 1 cleaning workflow. The raw NSW statistics CSV is cleaned once, then the cleaned dataset is used as the common input for each member's derived statistics.

The cleaning step is not an individual finding by itself. Its purpose is to make sure all members use the same cleaned dataset, so the results are comparable and are not affected by different column names, missing values, numeric types, or data shape.


In [ ]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
processed_task1_cleaned_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

cleaned_df = run_task1_cleaning(
    str(raw_task1_csv),
    str(processed_task1_cleaned_csv),
)

display(cleaned_df.head())
display(pd.DataFrame([{"rows": len(cleaned_df), "columns": len(cleaned_df.columns)}]))

## Individual Derived Statistics

This section only calls the statistics functions defined in `xfan0282_statistics.py`. Each function returns a `StatisticResult` with a statistic id, title, value, unit, and description.

The aim is to show the five derived statistics completed by this member. The full results stay in this notebook, while the group report only needs to select one or two findings that are most useful for the overall analysis.


In [ ]:
results = []
errors = []

for statistic_function in statistics_module.STATISTICS:
    try:
        result = statistic_function(cleaned_df)
    except NotImplementedError:
        continue
    except Exception as exc:
        errors.append({"function": statistic_function.__name__, "error": f"{type(exc).__name__}: {exc}"})
        continue
    results.append(result.to_dict())

statistics_df = (
    pd.DataFrame(results)
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 100)
display(
    statistics_df[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)

if errors:
    display(pd.DataFrame(errors))

## Explanation Notes

This section explains the five derived statistics for `xfan0282`. They are based on the shared cleaned dataset and cover dwelling structure, working pattern, commuting behaviour, and housing stress.

- `xfan0282-1`: The apartment share increased by 2.90 percentage points from 2011 to 2021, reaching 21.72% in 2021. This suggests that apartments became a larger part of occupied private dwellings in NSW, which can be read as a signal of higher-density housing.
- `xfan0282-2`: The work-from-home share grew 6.42 times from 2016 to 2021, rising from 4.82% to 30.98%. This is one of the strongest changes in the five statistics, and it may reflect a structural change in work patterns and commuting demand after COVID.
- `xfan0282-3`: The public transport commute share dropped by 11.98 percentage points from 2016 to 2021, falling from 15.98% to 4.00%. This can be interpreted together with the work-from-home growth, because both results point to a clear change in commuting behaviour.
- `xfan0282-4`: The commute distance gap across occupations was 5.30 km, with values ranging from 14.7 km to 20.0 km. This suggests that different occupation groups may face different commuting burdens, and that one average commute distance can hide these differences.
- `xfan0282-5`: In 2021, rent stress was 2.05 times mortgage stress. Rent stress was 35.5%, while mortgage stress was 17.3%. This gives one view of housing affordability pressure, and suggests that renters may face higher stress than mortgage holders.

The two findings that are most useful for the group report are `xfan0282-2` and `xfan0282-3`. Together, they show strong growth in working from home and a strong drop in public transport commuting. These findings can provide background for later discussion about urban resources, transport POIs, and accessibility.
